In [ ]:
!pip install yfinance scikit-learn seaborn tensorflow statsmodels

In [ ]:

import yfinance as yf
import pandas as pd 
import numpy as np 
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
import matplotlib.pyplot as plt
import seaborn as sn
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error
import joblib



In [ ]:
df = pd.read_csv(r'C:\Users\gerem\Desktop\projeto_tc_fiap_4\data\raw\data_2026_04_13.csv')

In [ ]:
df.columns = df.columns.get_level_values(0)
df = df.iloc[2:]

In [ ]:
df = df.rename(columns={'Price': 'data', 'Close': 'fechamento', 'High':'maxima', 'Low': 'minima', 'Open': 'abertura', 'Volume': 'volume'})


In [ ]:
df = df[df['data'] > '2021-01-01']

In [ ]:
df.head()

,data,fechamento,maxima,minima,abertura,volume
3979,2021-01-04,12.45572280883789,13.019030555367095,12.425900596844345,12.999148659362074,27749200
3980,2021-01-05,12.326491355895996,12.409330677825368,12.160812712037254,12.409330677825368,31908000
3981,2021-01-06,12.608150482177734,12.833473645055289,12.386141601593625,12.43584445974029,41133400
3982,2021-01-07,13.108492851257324,13.154882662548532,12.565065916628368,12.631337617341906,43757400
3983,2021-01-08,13.184709548950195,13.439854865727781,13.01903087031758,13.204590181408186,32480600


In [ ]:
df["data"] = pd.to_datetime(df["data"])

numeric_cols = ["fechamento", "maxima", "minima", "abertura", "volume"]
df[numeric_cols] = df[numeric_cols].astype(float)

In [ ]:
df.dtypes

data          datetime64[us]
fechamento           float64
maxima               float64
minima               float64
abertura             float64
volume               float64
dtype: object

In [ ]:
df['return'] = df['fechamento'].pct_change()
df['range'] = df['maxima'] - df['minima']
df['variacao'] = df['fechamento'] - df['abertura']
df['vol_10'] = df['return'].rolling(10).std()
df['vol_20'] = df['return'].rolling(20).std()
df['fechamento_1d'] = df['fechamento'].shift(1)
df['fechamento_2d'] = df['fechamento'].shift(2)
df['fechamento_3d'] = df['fechamento'].shift(3)
df['sma_5'] = df['fechamento'].rolling(5).mean()
df['sma_10'] = df['fechamento'].rolling(10).mean()
df['sma_20'] = df['fechamento'].rolling(20).mean()
df['ema_5'] = df['fechamento'].ewm(span=5).mean()
df['ema_10'] = df['fechamento'].ewm(span=10).mean()
df['mom_5'] = df['fechamento'] - df['fechamento'].shift(5)
df['mom_10'] = df['fechamento'] - df['fechamento'].shift(10)
df['target'] = df['fechamento'].shift(-1)

delta = df['fechamento'].diff()

gain = (delta.where(delta > 0, 0)).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()

rs = gain / loss
df['rsi'] = 100 - (100 / (1 + rs))

df = df.dropna()